# 3.5 Hiyerarşik İndeksleme

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/05-hierarchical-indexing.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Hierarchical Indexing

Şimdiye kadar esas olarak Pandas Series ve DataFrame nesnelerinde bir ve iki boyutlu veriye odaklandık. Çoğu zaman bir veya iki anahtardan fazla indekslenmiş daha yüksek boyutlu veri saklamak yararlıdır. Erken Pandas sürümleri Panel ve Panel4D sundu; pratikte hantal kaldılar. Daha yaygın desen, tek bir indekste birden fazla indeks düzeyi barındıran hiyerarşik indeksleme (çoklu indeksleme) kullanmaktır. Böylece yüksek boyutlu veri tanıdık bir ve iki boyutlu nesnelerde kompakt temsil edilir. (Pandas tarzı esnek indeksli gerçek N boyutlu diziler için Xarray paketine bakın.)

Bu bölümde MultiIndex nesnelerinin doğrudan oluşturulması, çoklu indeksli veride indeksleme, dilimleme ve istatistik, basit ile hiyerarşik gösterimler arası dönüşüm rutinleri ele alınır.

Standart içe aktarmalarla başlayalım:


In [ ]:
# import_pd_np.py
import pandas as pd
import numpy as np



## Çoklu İndeksli Series

İki boyutlu veriyi tek boyutlu Series içinde nasıl temsil edebileceğimizi düşünelim. Somut örnek: her noktanın metin ve sayısal anahtarı olan bir dizi.

### Kötü Yol

İki farklı yıldan eyalet verisi izlemek isteyelim. Mevcut araçlarla Python demetlerini anahtar olarak kullanmaya meyillisiniz:


In [ ]:
# pop_tuple_index.py
index = [('California', 2010), ('California', 2020),
         ('New York', 2010), ('New York', 2020),
         ('Texas', 2010), ('Texas', 2020)]
populations = [37253956, 39538223,
               19378102, 20201249,
               25145561, 29145505]
pop = pd.Series(populations, index=index)
pop



Bu şemayla seriyi demet indeksine göre dilimleyebilirsiniz:


In [ ]:
# pop_slice_tuple.py
pop[('California', 2020):('Texas', 2010)]



Kolaylık burada biter. 2010'daki tüm değerleri seçmek için dağınık (ve büyük veride yavaş) dönüşüm gerekir:


In [ ]:
# pop_filter_2010.py
pop[[i for i in pop.index if i[1] == 2010]]



İstenen sonuç gelir ama Pandas'ın sevdiğimiz dilimleme sözdizimi kadar temiz veya verimli değildir.

### Daha İyi Yol: Pandas MultiIndex

Neyse ki Pandas daha iyi bir yol sunar. Demet tabanlı indeksleme ilkel bir çoklu indekstir; MultiIndex istediğimiz işlemleri sağlar:


In [ ]:
# multiindex_from_tuples.py
index = pd.MultiIndex.from_tuples(index)



MultiIndex birden fazla düzey (burada eyalet ve yıl) ve her veri noktası için bu düzeyleri kodlayan etiketler temsil eder.


In [ ]:
# pop_reindex_multi.py
pop = pop.reindex(index)
pop



Series gösteriminin ilk iki sütunu çoklu indeks değerlerini, üçüncüsü veriyi gösterir. İlk sütunda boş girişler, bir üst satırla aynı değeri gösterir.

> **Not**
>

İkinci indeksi 2020 olan tüm verilere Pandas dilimleme ile erişelim:


In [ ]:
# pop_slice_2020.py
pop[:, 2020]



Sonuç, ilgilendiğimiz anahtarlarla tek indeksli bir Series'tir. Bu sözdizimi demet tabanlı çözümden çok daha kullanışlı ve verimlidir.

### MultiIndex Ek Boyut Olarak

Aynı veriyi indeks ve sütun etiketli basit DataFrame ile de saklayabilirdik. Pandas bu eşdeğerliği göz önünde bulundurur. unstack çoklu indeksli Series'i geleneksel DataFrame'e çevirir:


In [ ]:
# pop_unstack.py
pop_df = pop.unstack()
pop_df



stack ters işlemdir:


In [ ]:
# pop_stack.py
pop_df.stack()



Neden hiyerarşik indeksleme? İki boyutlu veriyi Series içinde çoklu indeksle yönettiğimiz gibi, üç veya daha fazla boyutu Series veya DataFrame'de yönetebiliriz. Her ek düzey ek bir veri boyutudur:


In [ ]:
# pop_df_under18.py
pop_df = pd.DataFrame({'total': pop,
                       'under18': [9284094, 8898092,
                                   4318033, 4181528,
                                   6879014, 7432474]})
pop_df



3.3 Pandas'ta İşlemler'deki ufunc'lar ve diğer işlevler hiyerarşik indekslerle de çalışır. 18 yaş altı oranı:


In [ ]:
# f_u18_unstack.py
f_u18 = pop_df['under18'] / pop_df['total']
f_u18.unstack()



Böylece yüksek boyutlu veriyi kolayca keşfedip işleyebiliriz.

## MultiIndex Oluşturma Yöntemleri

En doğrudan yol, oluşturucuya iki veya daha fazla indeks dizisi vermektir:


In [ ]:
# df_multiindex_ctor.py
df = pd.DataFrame(np.random.rand(4, 2),
                  index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                  columns=['data1', 'data2'])
df



MultiIndex oluşturma arka planda yapılır. Uygun demet anahtarlı sözlük verilirse Pandas otomatik MultiIndex kullanır:


In [ ]:
# series_from_dict_tuples.py
data = {('California', 2010): 37253956,
        ('California', 2020): 39538223,
        ('New York', 2010): 19378102,
        ('New York', 2020): 20201249,
        ('Texas', 2010): 25145561,
        ('Texas', 2020): 29145505}
pd.Series(data)



Bazen MultiIndex'i açıkça oluşturmak yararlıdır.

### Açık MultiIndex Oluşturucuları

pd.MultiIndex sınıfı oluşturucuları daha esnek indeks yapısı sağlar:


In [ ]:
# mi_from_arrays.py
pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]])



Her noktanın çoklu indeks değerlerini veren demet listesinden:


In [ ]:
# mi_from_tuples.py
pd.MultiIndex.from_tuples([('a', 1), ('a', 2), ('b', 1), ('b', 2)])



Tek indekslerin Kartezyen çarpımından:


In [ ]:
# mi_from_product.py
pd.MultiIndex.from_product([['a', 'b'], [1, 2]])



levels (her düzeydeki etiket listeleri) ve codes (bu etiketlere referans) ile:


In [ ]:
# mi_levels_codes.py
pd.MultiIndex(levels=[['a', 'b'], [1, 2]],
              codes=[[0, 0, 1, 1], [0, 1, 0, 1]])



Bunların hepsi Series veya DataFrame oluştururken index olarak veya reindex ile verilebilir.

### MultiIndex Düzey İsimleri

Düzeylere isim vermek için names argümanı veya sonradan names özniteliği:


In [ ]:
# pop_index_names.py
pop.index.names = ['state', 'year']
pop



Karmaşık veri kümelerinde indeks anlamını takip etmeye yardımcı olur.

### Sütunlar için MultiIndex

DataFrame'de satır ve sütun simetriktir; sütunlar da çoklu düzeyli olabilir. Örnek tıbbi veri:


In [ ]:
# hierarchical indices and columns
index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]],
                                   names=['year', 'visit'])
columns = pd.MultiIndex.from_product([['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
                                     names=['subject', 'type'])

# mock some data
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10
data += 37

# create the DataFrame
health_data = pd.DataFrame(data, index=index, columns=columns)
health_data



Temelde dört boyutlu veri: konu, ölçüm tipi, yıl, ziyaret. Üst düzey sütunla kişi adına göre indeksleyebiliriz:


In [ ]:
# health_guido.py
health_data['Guido']



## MultiIndex'te İndeksleme ve Dilimleme

Çoklu indeksli veride indeksleme sezgiseldir; indeksleri ek boyutlar gibi düşünmek yardımcı olur. Önce Series, sonra DataFrame.

### Çoklu İndeksli Series


In [ ]:
# pop_ref.py
pop



Birden fazla terimle tek elemana erişim:


In [ ]:
# pop_cal_2010.py
pop['California', 2010]



Kısmi indeksleme: yalnızca bir düzey — sonuç alt düzeyleri koruyan başka bir Series:


In [ ]:
# pop_california.py
pop['California']



Kısmi dilimleme, MultiIndex sıralı olduğunda mümkündür (Sıralı ve Sırasız İndeksler):


In [ ]:
# pop_loc_slice.py
pop.loc['California':'New York']



Sıralı indekslerde alt düzeyde kısmi indeksleme için ilk indekste boş dilim:


In [ ]:
# pop_partial_2010.py
pop[:, 2010]



3.2 Veri İndeksleme ve Seçimi'ndeki Boolean maske seçimi:


In [ ]:
# pop_bool_mask.py
pop[pop > 22000000]



Fancy indexing de çalışır:


In [ ]:
# pop_fancy.py
pop[['California', 'Texas']]



### Çoklu İndeksli DataFrame


In [ ]:
# health_data_ref.py
health_data



DataFrame'de sütunlar önceliklidir; çoklu indeksli Series sözdizimi sütunlara uygulanır:


In [ ]:
# health_guido_hr.py
health_data['Guido', 'HR']



Tek indeks durumunda olduğu gibi loc, iloc kullanılabilir (3.2):


In [ ]:
# health_iloc.py
health_data.iloc[:2, :2]



loc/iloc'a her indeks için demet verilebilir:


In [ ]:
# health_loc_bob_hr.py
health_data.loc[:, ('Bob', 'HR')]



Demet içinde dilim sözdizimi hatası verir:


```python
# health_bad_slice.py
health_data.loc[(:, 1), (:, 'HR')]  # SyntaxError — geçersiz dilim sözdizimi
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Python slice ile çözülebilir; bu bağlamda IndexSlice daha iyidir:


In [ ]:
# health_index_slice.py
idx = pd.IndexSlice
health_data.loc[idx[:, 1], idx[:, 'HR']]



> **Not**
>

Çoklu indeksli Series ve DataFrame ile etkileşimin en iyi yolu denemektir!

## Çoklu İndeksleri Yeniden Düzenleme

Çoklu indeksli veride bilgi koruyup düzeni değiştiren işlemler önemlidir. stack / unstack bunlardan biridir.

### Sıralı ve Sırasız İndeksler

Çoğu MultiIndex dilimleme işlemi indeks sıralı değilse başarısız olur. Sözlük sırası olmayan çoklu indeksli veri:


In [ ]:
# data_unsorted.py
index = pd.MultiIndex.from_product([['a', 'c', 'b'], [1, 2]])
data = pd.Series(np.random.rand(6), index=index)
data.index.names = ['char', 'int']
data



Kısmi dilim denemesi hata verir:


In [ ]:
# data_slice_error.py
try:
    data['a':'b']
except KeyError as e:
    print("KeyError", e)



Hata mesajından net olmasa da MultiIndex sıralı değildir. Kısmi dilimler için düzeyler sözlük (leksikografik) sırada olmalıdır. sort_index kullanın:


In [ ]:
# data_sort_index.py
data = data.sort_index()
data



Sıralandıktan sonra kısmi dilimleme beklenen gibi çalışır:


In [ ]:
# data_slice_ok.py
data['a':'b']



### İndeksleri Yığma ve Açma

Veri kümesini yığılmış çoklu indeksten iki boyutlu gösterime dönüştürmek mümkün; düzey belirtilebilir:


In [ ]:
# pop_unstack_l0.py
pop.unstack(level=0)



In [ ]:
# pop_unstack_l1.py
pop.unstack(level=1)



unstack'in tersi stack — orijinal seriyi geri alır:


In [ ]:
# pop_unstack_stack.py
pop.unstack().stack()



### İndeks Ayarlama ve Sıfırlama

İndeks etiketlerini sütunlara çevirmek için reset_index:


In [ ]:
# pop_reset_index.py
pop_flat = pop.reset_index(name='population')
pop_flat



Sütun değerlerinden MultiIndex oluşturmak için set_index:


In [ ]:
# pop_set_index.py
pop_flat.set_index(['state', 'year'])



Gerçek veri kümelerini keşfederken bu yeniden indeksleme desenlerinden biri en yararlılarındandır.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      İki düzeyli MultiIndex ile küçük bir Series oluşturup unstack ve stack deneyin:
          
      import pandas as pd
idx = pd.MultiIndex.from_product([['X','Y'], [1,2]])
s = pd.Series([10, 20, 30, 40], index=idx)
print(s)
print("\nunstack:\n", s.unstack())
print("\nstack:\n", s.unstack().stack())

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Sırasız çoklu indeks oluşturup sort_index() sonrası dilimlemeyi deneyin:
          
      import pandas as pd
import numpy as np
idx = pd.MultiIndex.from_product([['a','c','b'], [1,2]])
s = pd.Series(np.arange(6), index=idx)
try:
    print(s['a':'b'])
except KeyError:
    print("Sırasız dilimleme hata verdi")
print("\nSıralı:\n", s.sort_index()['a':'b'])

> **Not**
>

> **Not**
>

> **Not**
>
